# A categorically organized CAS in Lean

This notebook is the acceptance proof of the CasDsl vertical slice: a
computer algebra system whose **user-facing interfaces are organized by the
mathematical categories where operations first make sense** — not by
implementation classes, and not by backends.

Everything below runs in a persistent Lean 4 kernel
([lean-jupyter-kernel](https://github.com/dzackgarza/lean-jupyter-kernel)).
Three invariants to watch for:

1. **Backend-blind syntax.** You will never see a backend named in an
   expression. Some results below are computed by SageMath through a direct
   typed adapter — the *developer's* routing configuration decides that,
   and `#explain_route` will show it. The mathematics doesn't change.
2. **Category-owned methods.** `factor`, `det`, `annihilator`, `nth` are
   declared on categories; objects receive them by membership and by
   *subcategory inheritance*, never by forwarding code on a leaf class.
3. **Semantic availability ≠ computability.** A method that makes
   mathematical sense stays available even when no implementation route
   exists yet — execution then fails with a *structured capability gap*
   (an auditable developer backlog item), never a fake value and never a
   type error. The final cell demonstrates this deliberately.


## 1 · Trusted arithmetic and assertions

`assert` is an *operational* assertion in the ordinary CAS sense: the
predicate is computed and trusted, with a fourfold outcome
`true | false | unknown | error`. Only `true` lets the cell commit.
No Lean theorem is generated, and no certificate is required — this is a
CAS, not a proof obligation machine.


In [ ]:
assert 2 + 3 = 5

In [ ]:
assert 2 + 3 = 0 in ℤ/5

## 2 · Backend-blind factorization

`factor` is declared on the category of factorization-domain elements.
An integer receives it because `EuclideanElems(ℤ) ≤ FactorizationElems(ℤ)`
— the method arrives by **subcategory inheritance through the category
graph**, and is executed by whatever implementation the developer routed.


In [ ]:
let n := 360 in ℤ

In [ ]:
n.factor()

The expression above never mentioned a backend. The routing that chose one
is developer diagnostics, not mathematics:


In [ ]:
#explain_route n.factor()

## 3 · Polynomials, canonical embeddings, and calling a polynomial

`ℤ ⊆ ℚ` denotes the preferred canonical embedding — so `map p to ℚ[x]`
moves a polynomial along it without ceremony. And a polynomial can simply
be **called**: elaboration inserts evaluation through the preferred
compatible coefficient map. The mathematician writes `q(1)`, as on paper.


In [ ]:
let p(x) := x^3 - 2x + 1 in ℤ[x]

In [ ]:
let q := map p to ℚ[x]

In [ ]:
q.factor()

In [ ]:
assert q(1) = 0

## 4 · Exact matrix algebra

Matrix literals use row-semicolon syntax; `det` and `inverse` are methods
of the square-matrix category, computed exactly over `ℚ`.


In [ ]:
let M := [1, 2; 3, 4] in Mat₂(ℚ)

In [ ]:
M.inverse()

In [ ]:
assert M.det() = -2

## 5 · Subcategory inheritance, for real

`annihilator : Modules(ℤ) → Ideals(ℤ)` is declared **once**, on the parent
category. `F` below is declared in the *proper subcategory*
`SmallModules(ℤ)` — which contains **no forwarding declaration**. The
method arrives purely through the registered inclusion
`SmallModules ≤ Modules`. (The ascription is doing real semantic work:
`ℤ/4` *in a module category* means the ℤ-module ℤ/4, not the ring.)


In [ ]:
let F := ℤ/4 in SmallModules(ℤ)

In [ ]:
F.annihilator()

## 6 · Countable sets, ellipses, and indexing

Countability is mathematical structure — a monomorphism into ℕ — not a
backend capability. A *registered enumeration choice* labels elements, so
countable objects support `nth` (`X[k]`, 0-based) and `cardinality`.
Ellipsis literals are the exact Haskell-style progressions, nothing more.

The registered convention for `ℤ` is `0, 1, −1, 2, −2, …` — a documented,
revisitable choice, never a claim that ℤ is intrinsically ordered that way.


In [ ]:
let X := {0, 1, 2, ...}

In [ ]:
assert X = ℕ

In [ ]:
let Y := {0, 2, 4, ...}

In [ ]:
assert 8 ∈ Y

In [ ]:
assert 9 ∉ Y

In [ ]:
ℤ[3]

In [ ]:
X.cardinality()

## 7 · Semantic availability is not computability

`ℚ` is countable, so `nth` is *semantically* available on it — the category
layer says so, and no implementation hole is allowed to redefine the
mathematics (there is deliberately no `EnumerableCountableSet` category
here). But the developer has not yet registered an enumeration route for
`ℚ`. The audit surface shows the hole as structured backlog:


In [ ]:
#capability_gaps

So the next cell **fails on purpose** — with a structured
`NoImplementation` capability gap naming the method, the receiver
category, the presentation, and the routes considered. Not a parse error,
not a type error, and not a silent lie. This failing cell is part of the
proof.


In [ ]:
ℚ[3]